In [1]:
import os
import yaml
import json
import logging
from pyspark.ml.classification import GBTClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from sklearn.metrics import classification_report
from pyspark.sql.types import FloatType
from pyspark.sql.functions import udf, col


from utils.spark_session import get_spark_session

In [2]:
spark = get_spark_session(app_name="06-model-final")

In [3]:
# Paths
encoded_path = os.path.join("..", "data", "train_test", "train_encoded.parquet")
config_path = os.path.join("..", "src", "config", "feature_config.yaml")
params_path = os.path.join("..", "models", "tuning_best_params_gbt.json")
features_selected_path = os.path.join("..", "src", "features", "selected", "features_selected.yaml")
model_output_path = os.path.join("..", "models", "final_gbt_model")

In [4]:
with open(config_path, 'r') as f:
    feature_config = yaml.safe_load(f)
target_col = [k for k, v in feature_config.items() if isinstance(v, dict) and v.get("target")][0]

with open(features_selected_path, 'r') as f:
    selected_yaml = yaml.safe_load(f)
feature_cols = selected_yaml.get("support_random_forest", [])

In [5]:
train_df = spark.read.parquet(encoded_path)
train_df.show(5)

+--------------------+----------------+--------------------+
|            features|target_converted|          account_id|
+--------------------+----------------+--------------------+
|[3.07777777777777...|               0|0009655768c64bdeb...|
|[1.88472222222222...|               1|0009655768c64bdeb...|
|[2.2375E-4,22.0,1...|               1|0009655768c64bdeb...|
|[1.88333333333333...|               0|0009655768c64bdeb...|
|[1.70416666666666...|               1|0009655768c64bdeb...|
+--------------------+----------------+--------------------+
only showing top 5 rows



In [6]:
if target_col != "label":
    train_df = train_df.withColumnRenamed(target_col, "label")

In [7]:
# Load hyperparameters
with open(params_path, 'r') as f:
    best_params = json.load(f)

In [8]:
# Train and save model
gbt = GBTClassifier(
    labelCol="label",
    featuresCol="features",
    maxDepth=int(best_params["maxDepth"]),
    maxIter=int(best_params["maxIter"]),
    stepSize=float(best_params["stepSize"]),
    seed=96
)

final_model = gbt.fit(train_df.select("features", "label"))
final_model.write().overwrite().save(model_output_path)

print(f"Final GBT model trained and saved to: {model_output_path}")

Final GBT model trained and saved to: ..\models\final_gbt_model


In [9]:
predictions = final_model.transform(train_df.select("features", "label"))

evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)
auc = evaluator.evaluate(predictions)
print(f"AUC on training set: {auc:.4f}")

extract_prob_1 = udf(lambda v: float(v[1]), FloatType())
predictions = predictions.withColumn("prob_1", extract_prob_1(col("probability")))

pred_pd = predictions.select("prediction", "label", "prob_1").toPandas()

report = classification_report(
    y_true=pred_pd["label"],
    y_pred=pred_pd["prediction"],
    digits=4
)
print("\nClassification Report (Training Set):")
print(report)


AUC on training set: 0.9160

Classification Report (Training Set):
              precision    recall  f1-score   support

           0     0.8949    0.9308    0.9125     86719
           1     0.7153    0.6141    0.6608     24561

    accuracy                         0.8609    111280
   macro avg     0.8051    0.7724    0.7867    111280
weighted avg     0.8553    0.8609    0.8569    111280



In [10]:
spark.stop()